# Beyond Traditional Risk Scores: A Hybrid Epidemiology–Machine Learning Approach to Hypertension Prediction

**Authors:** O.K., O.A.  
**Dataset:** UCI Machine Learning Repository — Hypertension Dataset  
**Last updated:** April 2026

---

## Notebook Structure
1. Research Objective & Key Questions
2. Library Imports
3. Data Loading & Initial Exploration
4. Data Preprocessing — CORRECTED (leakage-aware)
5. Exploratory Data Analysis (EDA)
6. Model Training — Full Feature Set
7. Model Training — Leakage-Corrected Feature Set (lifestyle + demographic only)
8. Model Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)
9. Cross-Validation
10. Calibration Analysis
11. Feature Importance & SHAP
12. Model Export

## 1. Research Objective

Develop and evaluate machine learning models that predict the likelihood of hypertension based on **demographic, lifestyle, and clinical factors**.

### Key Research Questions
- Which factors contribute most to hypertension risk?
- Which ML model performs best for prediction?
- **Can lifestyle and demographic variables ALONE predict hypertension effectively?** *(leakage-corrected analysis)*
- How does model performance change when clinically proximal features (BP history, medications) are excluded?
- How early can we detect risk before clinical diagnosis?

## 2. Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Interpretability
import shap

# Model saving
import joblib

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully.')

## 3. Data Loading & Initial Exploration

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# Update this path to match your environment.
# For Google Colab: '/content/drive/MyDrive/YOUR_FOLDER/hypertension.csv'
# For local/GitHub: '../data/hypertension.csv'

DATA_PATH = '/content/drive/MyDrive/seyi/datasetss/hypertention.csv'  # <-- update as needed

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Basic summary ─────────────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicates ===')
print(f'Number of duplicate rows: {df.duplicated().sum()}')

print('\n=== Target Variable Distribution ===')
print(df['Has_Hypertension'].value_counts())
print(df['Has_Hypertension'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
# ── Numerical summary ─────────────────────────────────────────────────────────
df.describe().round(2)

## 4. Data Preprocessing — Leakage-Aware Design

### ⚠️ IMPORTANT: Feature Leakage Explanation

The original notebook included **two features that constitute data leakage** for the prediction task:

| Feature | Problem |
|---|---|
| `BP_History` | Contains 'Hypertension' as a category — this is essentially the outcome re-coded as a predictor |
| `Medication` | Contains antihypertensive drug classes (Beta Blocker, ACE Inhibitor, Diuretic) — patients taking these are already diagnosed |

Including these features produces near-ceiling accuracy (~99%) but a model that **cannot generalise** to undiagnosed individuals — the exact population we want to screen.

**Our approach:** We run TWO parallel analyses:
- **Model Set A (Full):** All features included — to reproduce prior results and confirm leakage
- **Model Set B (Corrected):** Only lifestyle + demographic features — the clinically meaningful prediction task

This comparison itself becomes a key finding of the paper.

In [ ]:
# ── Step 4a: Handle missing values ───────────────────────────────────────────
# Only Medication has missing values — impute with mode
df['Medication'].fillna(df['Medication'].mode()[0], inplace=True)

# Confirm no remaining missing values
assert df.isnull().sum().sum() == 0, 'Missing values remain after imputation!'
print('Missing values handled. No missing values remain.')

In [ ]:
# ── Step 4b: Define feature sets ─────────────────────────────────────────────

TARGET = 'Has_Hypertension'

# LEAKAGE features — excluded from corrected analysis
LEAKAGE_FEATURES = ['BP_History', 'Medication']

# Numerical features (same in both sets)
NUM_FEATURES = ['Age', 'Salt_Intake', 'Stress_Score', 'Sleep_Duration', 'BMI']

# Nominal categorical features (to be one-hot encoded)
# Full set includes leakage features
CAT_FEATURES_FULL = ['BP_History', 'Medication', 'Family_History', 'Exercise_Level', 'Smoking_Status']

# Corrected set — lifestyle + demographic only
CAT_FEATURES_CORRECTED = ['Family_History', 'Exercise_Level', 'Smoking_Status']

print('Feature sets defined.')
print(f'  Full feature set:      {NUM_FEATURES + CAT_FEATURES_FULL}')
print(f'  Corrected feature set: {NUM_FEATURES + CAT_FEATURES_CORRECTED}')

In [ ]:
# ── Step 4c: Encode target variable ──────────────────────────────────────────
# Map 'Yes' -> 1, 'No' -> 0 explicitly (avoid LabelEncoder ambiguity)
df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})

print('Target variable encoded: Yes=1, No=0')
print(df[TARGET].value_counts())

In [ ]:
# ── Step 4d: Build preprocessing pipelines ───────────────────────────────────
# NOTE: We use OneHotEncoder for all nominal categorical variables.
# The original notebook used LabelEncoder, which incorrectly implies
# ordinal relationships between categories (e.g. Beta Blocker < ACE Inhibitor).
# OneHotEncoder creates separate binary columns for each category.

def build_preprocessor(num_features, cat_features):
    """Build a ColumnTransformer that scales numerics and one-hot encodes categoricals."""
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])

preprocessor_full      = build_preprocessor(NUM_FEATURES, CAT_FEATURES_FULL)
preprocessor_corrected = build_preprocessor(NUM_FEATURES, CAT_FEATURES_CORRECTED)

print('Preprocessing pipelines built.')
print('  Numerical features:  StandardScaler (z-score normalisation)')
print('  Categorical features: OneHotEncoder (nominal — no ordinal assumption)')

In [ ]:
# ── Step 4e: Prepare X and y for both feature sets ───────────────────────────

# FULL feature set (includes leakage features — for comparison only)
X_full = df[NUM_FEATURES + CAT_FEATURES_FULL]
y      = df[TARGET]

# CORRECTED feature set (lifestyle + demographic only — main analysis)
X_corrected = df[NUM_FEATURES + CAT_FEATURES_CORRECTED]

# Train-test split — stratified to preserve class balance in both sets
RANDOM_STATE = 42
TEST_SIZE    = 0.20

X_full_train, X_full_test, y_train, y_test = train_test_split(
    X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

X_corr_train, X_corr_test, _, _ = train_test_split(
    X_corrected, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size: {X_full_train.shape[0]} | Test size: {X_full_test.shape[0]}')
print(f'Class balance in train — y=1: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Class balance in test  — y=1: {y_test.sum()} ({y_test.mean()*100:.1f}%)')

In [ ]:
# ── Step 4f: Fit preprocessors and transform ─────────────────────────────────
# CRITICAL: fit ONLY on training data, then transform both train and test.
# Fitting on the full dataset before splitting would cause data leakage.

X_full_train_proc = preprocessor_full.fit_transform(X_full_train)
X_full_test_proc  = preprocessor_full.transform(X_full_test)

X_corr_train_proc = preprocessor_corrected.fit_transform(X_corr_train)
X_corr_test_proc  = preprocessor_corrected.transform(X_corr_test)

print(f'Full feature matrix shape     — Train: {X_full_train_proc.shape} | Test: {X_full_test_proc.shape}')
print(f'Corrected feature matrix shape — Train: {X_corr_train_proc.shape} | Test: {X_corr_test_proc.shape}')

---
**✅ Step 1 Complete — Data leakage addressed.**

We now have two clean, preprocessed feature matrices:
- **Full set** (with `BP_History` + `Medication`) — reproduces prior high accuracy, confirms leakage
- **Corrected set** (lifestyle + demographic only) — the clinically generalisable prediction task

Key corrections made vs original notebook:
1. `LabelEncoder` → `OneHotEncoder` for nominal categoricals
2. Preprocessor fitted on **training data only** (prevents leakage)
3. Stratified train-test split ensures class balance
4. Target explicitly mapped `Yes=1, No=0`
5. Two parallel feature sets defined for comparison analysis

**Next: Step 2 — Exploratory Data Analysis (EDA)**